# 🚑 Ambulance Dispatch — GRPO Training with HF TRL + Unsloth

This notebook trains a small LLM to dispatch ambulances using GRPO (Group Relative Policy Optimization).

**Environment**: City-scale ambulance dispatch (OpenEnv — RFC 001–005 compliant)  
**Model**: `unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit` (free T4 compatible, ~15 GB VRAM)  
**Training**: GRPO via HuggingFace TRL  
**Reward**: 9-component RFC 004 Rubric  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CSNEHA20/Meta_PyTorch_OpenEnv_Hackathon/blob/main/notebooks/Ambulance_GRPO_Training.ipynb)

> **Runtime**: Free Colab T4 (15 GB). Training 100 steps takes ~15–20 min.


In [ ]:
# Cell 1 — Install dependencies
# Unsloth gives 2x faster GRPO training on free Colab T4.
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("trl>=0.12.0", "transformers>=4.40", "datasets", "torch", "matplotlib", "numpy", "networkx", "pydantic>=2.0")

try:
    pip("unsloth")
    USE_UNSLOTH = True
    print("✅ Unsloth installed — 4-bit quantised GRPO training enabled")
except Exception:
    USE_UNSLOTH = False
    print("⚠️  Unsloth not available — falling back to standard transformers")

print(f"USE_UNSLOTH = {USE_UNSLOTH}")


In [ ]:
# Cell 2 — Clone repo and set working directory
import os, subprocess, sys

REPO_URL = "https://github.com/CSNEHA20/Meta_PyTorch_OpenEnv_Hackathon.git"
REPO_DIR = "Ambulance-OpenENV"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
    print(f"✅ Cloned {REPO_URL}")
else:
    print(f"✅ Repo already present at {REPO_DIR}/")

if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")


In [ ]:
# Cell 3 — Imports and environment setup
import json, csv, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from env.environment import AmbulanceEnvironment
from env.models import ActionModel, AmbulanceState, Severity

ENV_CONFIG = {
    "n_ambulances": 2,
    "n_hospitals": 2,
    "max_steps": 30,
    "lambda_param": 0.3,
    "seed": 42,
}

env = AmbulanceEnvironment(ENV_CONFIG)
obs = env.reset()
print(f"✅ Env ready: {len(obs.ambulances)} ambulances, {len(obs.hospitals)} hospitals")
print(f"   Active emergencies at reset: {len(obs.emergencies)}")
print(f"   Step: {obs.step}  Done: {obs.done}")


In [ ]:
# Cell 4 — Prompt builder, response parser, and rollout helpers

SYSTEM_PROMPT = (
    "You are an expert emergency dispatch coordinator. "
    "Analyze the situation and dispatch the highest-priority unserved emergency "
    "using the nearest available idle ambulance, routing to the hospital with "
    "lowest occupancy and appropriate specialty. "
    'Respond ONLY with valid JSON: {"ambulance_id": <int|null>, "emergency_id": "<str>", "hospital_id": <int|null>}. '
    "Set ambulance_id to null and emergency_id to empty string for a no-op."
)

def build_prompt(obs):
    d = {
        "step": obs.step,
        "ambulances": [
            {"id": a.id, "node": a.node, "state": a.state.value, "eta": a.eta}
            for a in obs.ambulances
        ],
        "emergencies": [
            {"id": e.id, "node": e.node, "severity": e.severity.value, "time_remaining": e.time_remaining}
            for e in obs.emergencies if not e.assigned
        ],
        "hospitals": [
            {"id": h.id, "node": h.node, "occupancy": h.current_patients,
             "capacity": h.capacity, "specialty": h.specialty}
            for h in obs.hospitals
        ],
    }
    return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{json.dumps(d, indent=2)}\n<|assistant|>\n"

def parse_response(text, obs):
    try:
        s = text.find("{"); e = text.rfind("}") + 1
        if s == -1 or e == 0:
            return ActionModel(ambulance_id=None, emergency_id="", is_noop=True)
        p = json.loads(text[s:e])
        idle_ids = {a.id for a in obs.ambulances if a.state == AmbulanceState.IDLE}
        emg_ids  = {em.id for em in obs.emergencies if not em.assigned}
        hosp_ids = {h.id for h in obs.hospitals}
        if (p.get("ambulance_id") in idle_ids and
                p.get("emergency_id") in emg_ids and
                p.get("hospital_id") in hosp_ids):
            return ActionModel(
                ambulance_id=p["ambulance_id"],
                emergency_id=p["emergency_id"],
                hospital_id=p["hospital_id"],
            )
    except Exception:
        pass
    return ActionModel(ambulance_id=None, emergency_id="", is_noop=True)

def run_episode(generate_fn, seed=None):
    e = AmbulanceEnvironment(ENV_CONFIG)
    o = e.reset(seed=seed)
    total_r = 0.0
    rubric_totals = {}
    prompts_completions = []
    while not o.done:
        p = build_prompt(o)
        resp = generate_fn(p)
        action = parse_response(resp, o)
        o = e.step(action)
        total_r += o.reward
        if o.rubric:
            for k, v in o.rubric.model_dump().items():
                rubric_totals[k] = rubric_totals.get(k, 0.0) + v
        prompts_completions.append({"prompt": p, "completion": resp, "reward": o.reward})
    return total_r, rubric_totals, prompts_completions

print("✅ Prompt builder and rollout helpers defined.")


In [ ]:
# Cell 5 — Load model (Unsloth 4-bit if available, else standard transformers)
MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"  # fits free Colab T4 (15 GB)

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=2048,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "v_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
    print(f"✅ Unsloth model loaded: {MODEL_NAME}")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _base = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(_base, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(_base, trust_remote_code=True, device_map="auto")
    print(f"✅ Standard model loaded: {_base}")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"   Tokenizer pad_token: {tokenizer.pad_token!r}")


In [ ]:
# Cell 6 — Build GRPO dataset and configure GRPOTrainer
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

TRAINING_STEPS = 100  # ~15-20 min on Colab T4 free tier

# Reward function: reward valid JSON dispatch actions
def reward_fn(completions, prompts=None, **kwargs):
    rewards = []
    for c in completions:
        try:
            s = c.find("{"); e = c.rfind("}") + 1
            p = json.loads(c[s:e])
            has_keys = all(k in p for k in ["ambulance_id", "emergency_id", "hospital_id"])
            is_noop  = (p.get("ambulance_id") is None and p.get("emergency_id") == "")
            rewards.append(2.0 if has_keys and not is_noop else (0.5 if has_keys else -2.0))
        except Exception:
            rewards.append(-5.0)
    return rewards

# Build diverse prompt dataset from many environment seeds
dataset_prompts = []
for seed in range(TRAINING_STEPS * 2):
    e = AmbulanceEnvironment(ENV_CONFIG)
    o = e.reset(seed=seed)
    for _ in range(3):
        dataset_prompts.append({"prompt": build_prompt(o)})
        result = e.step(ActionModel(ambulance_id=None, emergency_id="", is_noop=True))
        o = result
        if o.done:
            break

train_dataset = Dataset.from_list(dataset_prompts)
print(f"✅ Dataset: {len(train_dataset)} prompts")

grpo_config = GRPOConfig(
    output_dir="/content/grpo_output",
    num_train_epochs=1,
    max_steps=TRAINING_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=5,
    save_steps=50,
    report_to="none",
    num_generations=4,
    max_new_tokens=128,
    temperature=0.7,
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    config=grpo_config,
    train_dataset=train_dataset,
    reward_funcs=reward_fn,
)
print("✅ GRPOTrainer configured. Ready to train.")


In [ ]:
# Cell 7 — Run GRPO training and save reward log
print(f"🚀 Starting GRPO training for {TRAINING_STEPS} steps...")
trainer.train()

log_history = trainer.state.log_history
reward_log = [
    (e["step"], e.get("train/reward", e.get("reward", 0.0)))
    for e in log_history if "step" in e
]

os.makedirs("/content", exist_ok=True)
with open("/content/grpo_rewards.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["step", "reward"])
    writer.writerows(reward_log)

print(f"✅ Training complete. Reward log: {len(reward_log)} entries")
trainer.save_model("/content/grpo_model")
print("✅ Model saved to /content/grpo_model")


In [ ]:
# Cell 8 — Evaluate trained vs greedy baseline and plot results
import torch

def make_generate_fn(mdl, tok, max_new=128):
    def fn(prompt):
        inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(
                **inputs, max_new_tokens=max_new, temperature=0.7,
                do_sample=True, pad_token_id=tok.pad_token_id
            )
        return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return fn

# Greedy baseline (rule-based, no LLM)
def greedy_baseline(prompt):
    try:
        obs_json = json.loads(prompt.split("<|user|>\n")[1].split("\n<|assistant|>")[0])
        ambs  = [a for a in obs_json["ambulances"] if a["state"] == "idle"]
        emgs  = sorted(obs_json["emergencies"],
                       key=lambda e: -({"CRITICAL": 3, "HIGH": 2, "NORMAL": 1}.get(e["severity"], 0)))
        hosps = sorted(obs_json["hospitals"], key=lambda h: h["occupancy"])
        if ambs and emgs and hosps:
            return json.dumps({"ambulance_id": ambs[0]["id"],
                               "emergency_id": emgs[0]["id"],
                               "hospital_id": hosps[0]["id"]})
    except Exception:
        pass
    return json.dumps({"ambulance_id": None, "emergency_id": "", "hospital_id": None})

N_EVAL = 10
generate_fn = make_generate_fn(model, tokenizer)
baseline_rewards, trained_rewards = [], []

print(f"Evaluating over {N_EVAL} episodes...")
for seed in range(N_EVAL):
    br, _, _ = run_episode(greedy_baseline, seed=seed)
    tr, _, _ = run_episode(generate_fn, seed=seed)
    baseline_rewards.append(br)
    trained_rewards.append(tr)
    print(f"  Episode {seed+1:2d}: greedy={br:.1f}  grpo={tr:.1f}")

print(f"\nGreedy baseline avg : {np.mean(baseline_rewards):.2f}")
print(f"GRPO trained avg    : {np.mean(trained_rewards):.2f}")
print(f"Improvement         : {np.mean(trained_rewards) - np.mean(baseline_rewards):+.2f}")

# ─── Plot 1: GRPO Training Reward Curve ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if reward_log:
    steps_r = [r[0] for r in reward_log]
    vals_r  = [r[1] for r in reward_log]
    window = 10
    ma = np.convolve(vals_r, np.ones(window) / window, mode="valid")
    axes[0].plot(steps_r, vals_r, alpha=0.35, color="#6366f1", linewidth=1.2, label="Per-step reward")
    axes[0].plot(steps_r[window-1:], ma, color="#6366f1", linewidth=2.5,
                 label=f"{window}-step moving avg")
    axes[0].set_title("GRPO Training Reward Curve", fontsize=13)
    axes[0].set_xlabel("Training Step")
    axes[0].set_ylabel("Reward")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

# ─── Plot 2: Before / After ────────────────────────────────────────────────────
x = np.arange(N_EVAL)
w = 0.35
axes[1].bar(x - w/2, baseline_rewards, w, label="Greedy Baseline", color="#6b7280", alpha=0.8)
axes[1].bar(x + w/2, trained_rewards,  w, label="GRPO Trained",    color="#10b981", alpha=0.8)
axes[1].axhline(np.mean(baseline_rewards), color="#6b7280", linestyle="--",
                label=f"Baseline avg={np.mean(baseline_rewards):.1f}")
axes[1].axhline(np.mean(trained_rewards),  color="#10b981", linestyle="--",
                label=f"GRPO avg={np.mean(trained_rewards):.1f}")
axes[1].set_title("Episode Reward: Greedy Baseline vs GRPO Trained", fontsize=13)
axes[1].set_xlabel("Evaluation Episode")
axes[1].set_ylabel("Total Episode Reward")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/grpo_comparison.png", dpi=150)
plt.show()
print("✅ Saved /content/grpo_comparison.png")
print("   Download via: files.download('/content/grpo_comparison.png')")
